In [1]:
import math
import pandas as pd

import geopandas as gpd
from geopy.geocoders import Nominatim

import folium 
from folium import Marker
from folium.plugins import MarkerCluster

In [2]:
def embed_map(m, file_name):
    from IPython.display import IFrame
    m.save(file_name)
    return IFrame(file_name, width='100%', height='500px')

In [3]:
# Load and preview Starbucks locations in California
starbucks = pd.read_csv("../datasets/geospatial-learn-course-data/starbucks_locations.csv")
starbucks.head()

,Store Number,Store Name,Address,City,Longitude,Latitude
0,10429-100710,Palmdale & Hwy 395,14136 US Hwy 395 Adelanto CA,Adelanto,-117.40,34.51
1,635-352,Kanan & Thousand Oaks,5827 Kanan Road Agoura CA,Agoura,-118.76,34.16
2,74510-27669,Vons-Agoura Hills #2001,5671 Kanan Rd. Agoura Hills CA,Agoura Hills,-118.76,34.15
3,29839-255026,Target Anaheim T-0677,8148 E SANTA ANA CANYON ROAD AHAHEIM CA,AHAHEIM,-117.75,33.87
4,23463-230284,Safeway - Alameda 3281,2600 5th Street Alameda CA,Alameda,-122.28,37.79


In [4]:
# How many rows in each column have missing values?
print(starbucks.isnull().sum())

# View rows with missing locations
rows_with_missing = starbucks[starbucks["City"]=="Berkeley"]
rows_with_missing

Store Number    0
Store Name      0
Address         0
City            0
Longitude       5
Latitude        5
dtype: int64


,Store Number,Store Name,Address,City,Longitude,Latitude
153,5406-945,2224 Shattuck - Berkeley,2224 Shattuck Avenue Berkeley CA,Berkeley,NaN,NaN
154,570-512,Solano Ave,1799 Solano Avenue Berkeley CA,Berkeley,NaN,NaN
155,17877-164526,Safeway - Berkeley #691,1444 Shattuck Place Berkeley CA,Berkeley,NaN,NaN
156,19864-202264,Telegraph & Ashby,3001 Telegraph Avenue Berkeley CA,Berkeley,NaN,NaN
157,9217-9253,2128 Oxford St.,2128 Oxford Street Berkeley CA,Berkeley,NaN,NaN


In [5]:
from tqdm.notebook import tqdm

# Create the geocoder
geolocator = Nominatim(user_agent="kaggle_learn")

def my_geocoder(row):
    try:
        point = geolocator.geocode(row).point
        return pd.Series({'Latitude': point.latitude, 'Longitude': point.longitude})
    except:
        return None

tqdm.pandas()

#berkley_locations = rows_with_missing.apply(lambda x: my_geocoder(x['Address']), axis=1)
berkley_locations = rows_with_missing.progress_apply(lambda x: my_geocoder(x['Address']), axis=1)
starbucks.update(berkley_locations)

display(berkley_locations)
starbucks[starbucks["City"] == 'Berkeley']

  0%|          | 0/5 [00:00<?, ?it/s]

,Latitude,Longitude
153,37.868839,-122.268230
154,37.891477,-122.280013
155,37.880907,-122.269679
156,37.855903,-122.259406
157,37.870253,-122.266095


,Store Number,Store Name,Address,City,Longitude,Latitude
153,5406-945,2224 Shattuck - Berkeley,2224 Shattuck Avenue Berkeley CA,Berkeley,-122.268230,37.868839
154,570-512,Solano Ave,1799 Solano Avenue Berkeley CA,Berkeley,-122.280013,37.891477
155,17877-164526,Safeway - Berkeley #691,1444 Shattuck Place Berkeley CA,Berkeley,-122.269679,37.880907
156,19864-202264,Telegraph & Ashby,3001 Telegraph Avenue Berkeley CA,Berkeley,-122.259406,37.855903
157,9217-9253,2128 Oxford St.,2128 Oxford Street Berkeley CA,Berkeley,-122.266095,37.870253


In [6]:
m_2 = folium.Map(location=[37.88,-122.26], zoom_start=13)

# Add a marker for each Berkeley location
for idx, row in starbucks[starbucks["City"]=='Berkeley'].iterrows():
    Marker([row['Latitude'], row['Longitude']], popup=row['Store Name']).add_to(m_2)

embed_map(m_2, 'mgd_q2.html')

In [7]:
from IPython.display import display, Markdown

content = f"""
**Q**: **Considering only the five locations in Berkeley, how many of the (latitude, longitude) locations seem 
potentially correct (are located in the correct city)?**<br>
**A**: All five locations appear to be correct!
"""
display(Markdown(content))


**Q**: **Considering only the five locations in Berkeley, how many of the (latitude, longitude) locations seem 
potentially correct (are located in the correct city)?**<br>
**A**: All five locations appear to be correct!


In [8]:
CA_COUNTIES_DATA = "../datasets/geospatial-learn-course-data/CA_county_boundaries/CA_county_boundaries.shp"
EPSG_4326 = 'epsg:4326'

CA_counties = gpd.read_file(CA_COUNTIES_DATA)
CA_counties.crs = EPSG_4326
CA_counties.head()

,GEOID,name,area_sqkm,geometry
0,6091,Sierra County,2491.995494,"POLYGON ((-120.6556 39.69357, -120.65554 39.69..."
1,6067,Sacramento County,2575.258262,"POLYGON ((-121.18858 38.71431, -121.18732 38.7..."
2,6083,Santa Barbara County,9813.817958,"MULTIPOLYGON (((-120.58191 34.09856, -120.5822..."
3,6009,Calaveras County,2685.626726,"POLYGON ((-120.63095 38.34111, -120.63058 38.3..."
4,6111,Ventura County,5719.321379,"MULTIPOLYGON (((-119.63631 33.27304, -119.6360..."


In [9]:
CA_POPULATION_DATA = "../datasets/geospatial-learn-course-data/CA_county_population.csv"
CA_HIGHEARNERS_DATA = "../datasets/geospatial-learn-course-data/CA_county_high_earners.csv"
CA_MEDIANAGE_DATA = "../datasets/geospatial-learn-course-data/CA_county_median_age.csv"
GEOID = "GEOID"

CA_pop = pd.read_csv(CA_POPULATION_DATA, index_col=GEOID)
CA_high_earners = pd.read_csv(CA_HIGHEARNERS_DATA, index_col=GEOID)
CA_median_age = pd.read_csv(CA_MEDIANAGE_DATA, index_col=GEOID)

In [10]:
# Join the CA_counties GeoDataFrame with CA_pop, CA_high_earners, and CA_median_age.

#cols_to_add = CA_pop.join([CA_high_earners, CA_median_age]).reset_index()
#CA_stats = CA_counties.merge(cols_to_add, on="GEOID")

CA_stats = (CA_counties
            .merge(CA_pop, on=GEOID)
            .merge(CA_high_earners, on=GEOID)
            .merge(CA_median_age, on=GEOID))
CA_stats.head()

,GEOID,name,area_sqkm,geometry,population,high_earners,median_age
0,6091,Sierra County,2491.995494,"POLYGON ((-120.6556 39.69357, -120.65554 39.69...",2987,111,55.0
1,6067,Sacramento County,2575.258262,"POLYGON ((-121.18858 38.71431, -121.18732 38.7...",1540975,65768,35.9
2,6083,Santa Barbara County,9813.817958,"MULTIPOLYGON (((-120.58191 34.09856, -120.5822...",446527,25231,33.7
3,6009,Calaveras County,2685.626726,"POLYGON ((-120.63095 38.34111, -120.63058 38.3...",45602,2046,51.6
4,6111,Ventura County,5719.321379,"MULTIPOLYGON (((-119.63631 33.27304, -119.6360...",850967,57121,37.5


In [11]:
# Create a "density" column with the population density.
CA_stats["density"] = CA_stats["population"] / CA_stats["area_sqkm"]

In [12]:
"""
create a GeoDataFrame sel_counties that contains a subset of the rows (and all of the columns) from the 
CA_stats GeoDataFrame. In particular, select counties where:

there are at least 100,000 households making $150,000 per year,
the median age is less than 38.5, and
the density of inhabitants is at least 285 (per square kilometer).
Additionally, selected counties should satisfy at least one of the following criteria:

there are at least 500,000 households making $150,000 per year,
the median age is less than 35.5, or
the density of inhabitants is at least 1400 (per square kilometer).
"""
sel_counties = CA_stats[(
    (CA_stats.high_earners > 1e5) & 
    (CA_stats.median_age < 38.5) & 
    (CA_stats.density > 285) & 
    ((CA_stats.high_earners > 5e5) | 
     (CA_stats.median_age < 35.5) | 
     (CA_stats.density > 1400))
)]

sel_counties

,GEOID,name,area_sqkm,geometry,population,high_earners,median_age,density
5,6037,Los Angeles County,12305.376879,"MULTIPOLYGON (((-118.66761 33.47749, -118.6682...",10105518,501413,36.0,821.227834
8,6073,San Diego County,11721.342229,"POLYGON ((-117.43744 33.17953, -117.44955 33.1...",3343364,194676,35.4,285.237299
10,6075,San Francisco County,600.588247,"MULTIPOLYGON (((-122.60025 37.80249, -122.6123...",883305,114989,38.3,1470.733077


In [13]:
EPSG_4326 = 'epsg:4326'
starbucks_gdf = gpd.GeoDataFrame(starbucks, crs=EPSG_4326,
                                 geometry=gpd.points_from_xy(starbucks.Longitude, starbucks.Latitude))
starbucks_gdf.shape

(2821, 7)

In [14]:
# How many stores are in the selected counties?
num_stores = gpd.sjoin(starbucks_gdf, sel_counties).shape[0]
print(f"There are {num_stores} in the selected counties")

There are 1043 in the selected counties


In [15]:
m_6 = folium.Map(location=[37,-120], zoom_start=6)
mc = MarkerCluster()

selected_locations = gpd.sjoin(starbucks_gdf, sel_counties)
selected_locations.dropna(axis=0, inplace=True)
display(selected_locations.shape)

# Show selected store locations
for idx, row in selected_locations.iterrows():
    mc.add_child(folium.Marker([row['Latitude'], row['Longitude']]))

m_6.add_child(mc)
embed_map(m_6, 'mgd_q6.html')

(1043, 15)